# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

In [1]:
import dotenv

print("dotenv works")

dotenv works


---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response

In [4]:
response = ask_llm(
    "What is Artificial Intelligence?"
)

print(response.choices[0].message.content)

print("\nTOKEN USAGE:")
print(response.usage)

**Artificial Intelligence (AI)** is a field of computer science that focuses on creating machines and systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, experiences, and interactions, enabling them to improve their performance over time.
2. **Reasoning**: AI systems can draw inferences, make decisions, and solve problems using logical and analytical approaches.
3. **Problem-solving**: AI systems can identify and address complex problems, often using novel and innovative solutions.
4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.

The primary goal of AI is to create systems that can:

1. **Simulate human thought processes**: AI systems aim to replicate human-like intelligence, using algorithms and data to make decisions and take actions.
2. **Automate tasks**: AI systems can perform tasks that are repetitive, mundane, or require human-like inte

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [the system role shows the ai how to behave,respond and act during conversation by giving it instructions that guide the ai throughout conversations.Example a system message could tell the ai to act as a microfinance loan officer and provide or give information that are only factual here the user role contains just the actual task the user wanst to perform, such as for example summarizing a customers loan application.
tokens are units of texts that are processed by ai, being either a full word,part of the full word,punctuation,character or even a space.Api providers normally charge based on usage of tokens because computational costs normally depend on the amount of text being processed and generated not per number of text requests.

### Part 1.2 — Temperature: the randomness dial

In [7]:
question = "Suggest a name for a savings product for market traders in Accra."

print("TEMPERATURE = 0.0")
print("=" * 60)

for i in range(5):
    response = ask_llm(
        question,
        temperature=0.0
    )

    print(f"\nRun {i+1}:")
    print(response.choices[0].message.content)

print("\n\nTEMPERATURE = 1.2")
print("=" * 60)

for i in range(5):
    response = ask_llm(
        question,
        temperature=1.2
    )

    print(f"\nRun {i+1}:")
    print(response.choices[0].message.content)

TEMPERATURE = 0.0

Run 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could encourage market traders to save and collect their earnings.
5. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
6. **Kae Dzi**: "Kae Dzi" is a Ghanaian phrase that means "save for the future". This name could appeal to market traders who are looking to plan for their future and se

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [the outputs were very consistent and predictable from the ai when the temperature was at 0.0 as the answers became very predictable, giving very similar answers and suggestions when same or similar questions were asked.
at temperature of 1.2, the generated answers were more varied,creative and diverse.

however for loan decision support systems the lower the temperature the better the output, so the low temperature is better as with loan applications the application and system needs to be consistent and very reliable. higher temperatures are better for task where creativity is needed.]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [9]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [10]:
# =========================
# SUMMARY PROMPT V1
# =========================

SUMMARY_PROMPT_V1 = "Summarize this loan application:"


def summarize_v1(letter_text):
    response = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
    )

    return response.choices[0].message.content


print("===== L002 : V1 =====")
print(summarize_v1(LETTERS["L002"]))

print("\n" + "=" * 100 + "\n")

print("===== L006 : V1 =====")
print(summarize_v1(LETTERS["L006"]))


# =========================
# SUMMARY PROMPT V2
# =========================

SUMMARY_SYSTEM_PROMPT = """
You are an assistant to a microfinance loan officer.

Requirements:
- Produce a factual and neutral summary.
- Use only information provided in the application.
- Do not invent or assume information.
- Limit the summary to 3-4 sentences.
- Mention the applicant, loan amount, purpose,
  repayment information, and notable strengths or risks.
"""


def summarize_v2(letter_text):
    response = ask_llm(
        f"Summarize this loan application:\n\n{letter_text}",
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        temperature=0
    )

    return response.choices[0].message.content


print("\n\n===== L002 : V2 =====")
print(summarize_v2(LETTERS["L002"]))

print("\n" + "=" * 100 + "\n")

print("===== L006 : V2 =====")
print(summarize_v2(LETTERS["L006"]))


# =========================
# SIDE-BY-SIDE COMPARISON
# =========================

for letter_id in ["L002", "L006"]:
    print("\n" + "=" * 120)
    print(f"{letter_id} - VERSION 1")
    print("=" * 120)
    print(summarize_v1(LETTERS[letter_id]))

    print("\n")
    print("=" * 120)
    print(f"{letter_id} - VERSION 2")
    print("=" * 120)
    print(summarize_v2(LETTERS[letter_id]))

===== L002 : V1 =====
Kwame Boateng, a commercial driver, is applying for a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He expects his business to improve after the festive season and is willing to repay the loan when his finances allow, but he currently has no collateral to offer.


===== L006 : V1 =====
Here is a summary of Kofi's loan application:

* Amount: GHS 50,000
* Purpose: To start three businesses - a car washing business, a provision shop, and a phone import business from Dubai
* Repayment term: 1 year
* Collateral: None
* Qualifications: Kofi is 22 years old, claims to be "full of energy" and "business-minded" based on his friends' opinions, but has no prior business experience.

Overall, the application is based on Kofi's enthusiasm and perceived potential, but lacks concrete plans, experience, and security for the loan.


===== L002 : V2 =====
Kwame Boateng, a commercial driver, has applied for a GHS 25,000 loan to repair his trotro engi

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [with the v1 prompt the summaries werent very well structured due to how simple the v1 prompt was. hence the summaries were quite long and important information and details werent emphasized as much or highlighted clearly. the v2 prompt produced much better results, as it emphasized and focused on the more imporantaspects of the loan officer such as the loan amount, risk involved, repayment plan and reason for the loan aswell.

with "no invented details" it restricts the ai to only using actual information provided by the appicant.this is essential as financial decsions should only be based on information provided by the applicants hence if the model creates or makes up its own information it could leadd to poor lending decisions, when the ai makes information not provided or creates information that doesnt exist this is known as hallucination in ai and llms.]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [11]:
import json
import pandas as pd

EXTRACT_PROMPT = """
You are an information extraction system.

Extract the requested information and return ONLY a valid JSON object.

Use EXACTLY this schema:

{
  "applicant_name": "",
  "amount_ghs": 0,
  "purpose": "",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": null
}

Rules:
- Return ONLY JSON.
- Do not include explanations.
- Do not include markdown.
- If a value is missing, use null.
- Do not guess.

Example:

Letter:
Dear Manager,
My name is Ama Owusu. I run a vegetable farm and request GHS 5,000 to buy irrigation equipment.
My farm makes about GHS 700 profit each month.
I will repay over 10 months.
My brother will act as guarantor.

JSON:
{
  "applicant_name": "Ama Owusu",
  "amount_ghs": 5000,
  "purpose": "buy irrigation equipment",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}
"""


def extract_fields(letter_text):
    
    response = ask_llm(
        f"{EXTRACT_PROMPT}\n\nLetter:\n{letter_text}",
        temperature=0
    )

    raw_output = response.choices[0].message.content.strip()

    raw_output = raw_output.replace("```json", "")
    raw_output = raw_output.replace("```", "")
    raw_output = raw_output.strip()

    try:
        data = json.loads(raw_output)
        return data

    except Exception as e:
        print("Warning: JSON parse failed")
        print(raw_output)
        return None


# Extract all six letters

records = []

for letter_id, letter_text in LETTERS.items():

    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        records.append(extracted)


df = pd.DataFrame(records)

df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [1.the few shot example shouldn't come from one of the six letters being tested and processed as if one of the test letters are used again as an example it would lead to bias results as the model would already have the information from test data, making the evaluation less accurate. Hence using other new examples which haven't been used before would allow the model to actually learn.
 
2. "use null, do not guess" is important cause it tells the ai not to just create and makeup information when information is missing. without this instruction thr model may become unreliable and less accurate as it ay fill in fields that were never used/stated in the letter.
 
3.because we want the ai to be consistent,predictable and there to be repeatability the most apprpopiate temperature is 0 for extractrion.higher temperatures are for task where creativity is needed they produce more diverse, original answers.
]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [13]:
import json

# ==========================
# PART 3.3 DECISION SUPPORT
# ==========================

BRIEF_PROMPT = """
You are assisting a human microfinance loan officer.

Your role is to provide decision support, NOT make decisions.

IMPORTANT RULES:
- Do NOT approve or reject applications.
- Use only information found in the loan application and extracted JSON.
- Do not invent information.
- Stay factual and neutral.
- Final lending decisions must always be made by a human.

Output EXACTLY these sections:

1. Strengths
   - Bullet points grounded in the application

2. Risks / Red Flags
   - Bullet points grounded in the application

3. Missing Information
   - Information the officer should request before making a decision

4. Suggested Next Step
   - One appropriate next step such as:
     * invite for interview
     * request supporting documents
     * request financial records
     * flag for senior review
"""


def generate_brief(letter_text, extracted_json):
    
    prompt = f"""
Loan Application:

{letter_text}

Extracted Information:

{json.dumps(extracted_json, indent=2)}

Prepare the decision-support brief.
"""

    response = ask_llm(
        prompt,
        system_prompt=BRIEF_PROMPT,
        temperature=0
    )

    return response.choices[0].message.content


# Generate briefs for all six letters

briefs = {}

for letter_id, letter_text in LETTERS.items():

    extracted = extract_fields(letter_text)

    briefs[letter_id] = generate_brief(
        letter_text,
        extracted
    )


# Print three example applications

print("===== L001 =====\n")
print(briefs["L001"])

print("\n" + "=" * 100 + "\n")

print("===== L002 =====\n")
print(briefs["L002"])

print("\n" + "=" * 100 + "\n")

print("===== L006 =====\n")
print(briefs["L006"])

===== L001 =====

1. Strengths
   - The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
   - She has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution.
   - The applicant has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods.
   - She has a guarantor, her sister, who is a teacher, potentially providing an added layer of security for the loan.
   - The applicant's current stall makes a significant profit of GHS 900 each month, which could help in repaying the loan.

2. Risks / Red Flags
   - The loan amount of GHS 8,000 is substantial compared to the applicant's monthly profit, which might pose a risk if the expansion into frozen foods does not generate enough additional income.
   - The repayment plan of GHS 450 monthly over 20 months needs to be carefully evaluated to ensure it is realistic a

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [1.with LOO3 the system highlighted many strengths like,business registration, high monthly profits, supporting documents, collateral, and a realistic repayment plan. and with l006 it also highlighted many major risks like including lack of business history, no clear source of income, no collateral, and multiple proposed businesses with little supporting evidence. these show that the model and system's able to identify merits and demerits in each loan case.

2.to prevent the ai from making mistakes or missing important information  we revented the model from giving final "approve" or "reject" decisions.From a more pratical point of view loan decisions must involve human judgment which can analyse different perspectives and make the final decision responsibly and ethically, ai's sould only as act a form of support in decision making.
---]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [1717e26]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [14]:
FIELDS = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

evaluation_rows = []

for field in FIELDS:

    row = {"Field": field}

    correct_count = 0

    for letter_id in ["L001", "L003", "L006"]:

        extracted_row = df[df["letter_id"] == letter_id].iloc[0]

        extracted_value = extracted_row[field]
        gold_value = GOLD[letter_id][field]

        if field == "applicant_name":
            match = str(extracted_value).strip().lower() == str(gold_value).strip().lower()

        else:
            match = extracted_value == gold_value

        row[letter_id] = "✓" if match else "✗"

        if match:
            correct_count += 1

    row["Accuracy"] = round(correct_count / 3, 2)

    evaluation_rows.append(row)

accuracy_df = pd.DataFrame(evaluation_rows)

accuracy_df

,Field,L001,L003,L006,Accuracy
0,applicant_name,✓,✓,✓,1.00
1,amount_ghs,✓,✓,✓,1.00
2,purpose,✗,✗,✗,0.00
3,monthly_profit_ghs,✓,✓,✗,0.67
4,has_collateral_or_guarantor,✓,✓,✓,1.00
5,repayment_months,✓,✓,✓,1.00


### Part 4.2 — Reliability: is the system consistent?

In [15]:
import json

# --------------------------
# Temperature = 0
# --------------------------

temp0_results = []

for i in range(5):
    result = extract_fields(LETTERS["L004"])

    if result is not None:
        temp0_results.append(
            json.dumps(result, sort_keys=True)
        )

temp0_valid_json = len(temp0_results)
temp0_unique_outputs = len(set(temp0_results))


# --------------------------
# Temperature = 1.0
# --------------------------

temp1_results = []

for i in range(5):

    response = ask_llm(
        f"{EXTRACT_PROMPT}\n\nLetter:\n{LETTERS['L004']}",
        temperature=1.0
    )

    raw_output = response.choices[0].message.content.strip()

    raw_output = raw_output.replace("```json", "")
    raw_output = raw_output.replace("```", "")

    try:
        parsed = json.loads(raw_output)

        temp1_results.append(
            json.dumps(parsed, sort_keys=True)
        )

    except:
        pass

temp1_valid_json = len(temp1_results)
temp1_unique_outputs = len(set(temp1_results))


# --------------------------
# Report Results
# --------------------------

print("TEMPERATURE = 0")
print("Valid JSON outputs:", temp0_valid_json, "/ 5")
print("Unique outputs:", temp0_unique_outputs)

print("\n" + "-" * 50 + "\n")

print("TEMPERATURE = 1.0")
print("Valid JSON outputs:", temp1_valid_json, "/ 5")
print("Unique outputs:", temp1_unique_outputs)


TEMPERATURE = 0
Valid JSON outputs: 5 / 5
Unique outputs: 1

--------------------------------------------------

TEMPERATURE = 1.0
Valid JSON outputs: 5 / 5
Unique outputs: 2


### Part 4.3 — Hallucination probing

In [16]:
# =====================================
# TEST 1: Missing Information Question
# =====================================

test1 = ask_llm(
    f"""
Loan Application:

{LETTERS['L001']}

Question:
What is the applicant's credit score?
""",
    temperature=0
)

print("TEST 1 OUTPUT")
print(test1.choices[0].message.content)

print("\n" + "=" * 80 + "\n")


# =====================================
# TEST 2: Irrelevant Input
# =====================================

weather_report = """
Today's weather in Accra is sunny with occasional clouds.
Temperatures will range between 25 and 32 degrees Celsius.
Light winds are expected throughout the day.
"""

test2 = extract_fields(weather_report)

print("TEST 2 OUTPUT")
print(test2)

TEST 1 OUTPUT
The applicant's credit score is not explicitly mentioned in the loan application. However, based on the information provided, we can infer that the applicant has a good credit history. Here's why:

* The applicant has been consistently saving with the susu scheme for 2 years and has never missed a contribution, which shows discipline and responsibility.
* The applicant has a stable business with a monthly profit of GHS 900, which suggests a reliable income stream.
* The applicant has a guarantor, a teacher, who is likely to have a stable income and can vouch for the applicant's creditworthiness.
* The applicant has a clear repayment plan, proposing to repay GHS 450 monthly over 20 months, which is approximately 50% of their monthly profit.

Overall, while the credit score is not explicitly mentioned, the applicant's credit history and financial discipline suggest that they may have a good credit score. However, the actual credit score would depend on various factors, incl

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [1. Perfect prediction was obtained for ‘applicantname’, ‘amountghs’, ‘hascollateralorguarantor’ and ‘repaymentmonths’ fields; “purpose” was the most difficult since the language and formulation generated was different to that in the gold data but the meaning was similar; and there were some inconsistencies in “monthlyprofitghs” due to how the model handled null and NaN values.

2. The reliability study showed that lower temperatures leads to more stable prediction. At temperature zero we had all five runs producing correct JSON and even correct and exactly similar output. However, if you increase temperature to 1.0 you would still get correct JSON but have different outputs coming through. It confirms the model works best with low temperature as they should produce very robust predictions that can be useful in making business decisions for production systems.

3. In “Test1” the system hallucinated partial responses but was able to correctly point out that the credit card number was not provided and then went on to draw its conclusion based on its own assumptions. If hallucinations may cause loss of significant damage they need to be controlled or limited through stronger prompts, implementing validation logic for the model or forcing a human to review every single decision.
]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [1. Business owners that run highly successful but have poor English writing ability may be adversely affected if lending decisions are fully automated and the system reads poor English as an indicator of poor business acumen. This would penalize applicants whose businesses are actually successful despite their low literacy or low English proficiency.

2. Personal and financial data would be submitted through loan application letters as these inherently contains sensitive information. Transmitting this sensitive information to a third party API hosted in an overseas location poses a variety of privacy, security, and compliance issues. I would look into aspects of data retention, security protocols employed, compliance certifications obtained, encryption standards, and specific legal requirements related to data protection and cross-border transfer of data.

3. I would implement two security measures:

- Human review required prior to any decision is made concerning whether or not to provide lending.
    - Thorough logs and audit trails of system recommendations for investigation in the event of dispute.]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [
1. Prompting as engineering:

It is analogous to hyperparameter tuning in that both involve the processes of trial-and-error, measurement, and evaluation to optimize outcomes. While hyperparameter tuning alters the behavior of models by adjusting numerical parameters, prompt engineering does it by defining instructions, rules, samples, and by assigning tasks.

2. Trust:

I would not trust it to run fully without human intervention. The test was the most indicative regarding my trust level due to the reasoning the model used in the hallucination test was a leap beyond what the given application presented. This implies that human oversight is still required.

3. Cost & Scale:

The one API prompt alone utilized about 459 tokens. At an estimated average of 500 tokens per call, considering 3 calls per application (summarization, extraction, decision support) this means it would cost about 1.5 million tokens per month to process 1000 applications this way. Therefore the cost, rate limits and token limits of the provider become critically relevant.

4. Look back at the class:

I think accessing a LLM API is the most practical path to taking advantage of the powerful language abilities LLM models offer compared to creating custom models. The time, expense, and large data sets needed to train models personally are prohibitive except for when certain circumstances justify custom models for a company.]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.